In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
import numpy as np
from scipy.stats import qmc
import matplotlib.pyplot as plt
from matplotlib import cm
import os
import json
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Callable

# ==================== 设备配置 ====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
print(f"使用设备: {device}")

# ==================== 几何参数 ====================
d0 = 0.05  # 内管直径（m）
d1 = 0.13  # 外壳直径（m）
r0 = d0 / 2  # 内管半径
r1 = d1 / 2  # 外壳半径
L_char = d1 - d0  # 特征长度：环形域宽度 (m)
area_total = np.pi * (r1**2 - r0**2)  # 总面积

# ==================== 材料参数 ====================
# PCM（石蜡）- 来自论文Table 1
rho_s = 880.0      # 固相密度 (kg/m³)
rho_l = 760.0      # 液相密度 (kg/m³)
cp_s = 2180.0      # 固相定压比热容 (J/(kg·K))
cp_l = 2390.0      # 液相定压比热容 (J/(kg·K))
lambda_s = 0.4     # 固相导热系数 (W/(m·K))
lambda_l = 0.15    # 液相导热系数 (W/(m·K))
mu_l = 0.001       # 液相粘度 (kg/(m·s))
L = 255000.0       # 相变潜热 (J/kg)
Tpc = 316.15       # 相变温度 (K)
DeltaT = 6.0       # 相变温度区间 (K)
alpha = 1.0e-4     # 体膨胀系数 (1/K)
g = 9.81           # 重力加速度 (m/s²)

# 高导热材料（铜）
rho_Cu = 8960.0    # 密度 (kg/m³)
lambda_Cu = 400.0  # 导热系数 (W/(m·K))
cp_Cu = 385.0      # 定压比热容 (J/(kg·K))
mu_Cu = 1e10       # 铜为固体，粘度取极大值抑制流动

# 外壳（铝） - 边界条件用
rho_Al = 2719.0
lambda_Al = 202.4
cp_Al = 879.0

# ==================== 计算特征尺度 ====================
def compute_characteristic_scales():
    """基于物理分析计算特征尺度"""
    # 参考热扩散率（液相）
    alpha_ref = lambda_l / (rho_l * cp_l)
    
    # 自然对流特征速度（基于瑞利数）
    nu_ref = mu_l / rho_l  # 液相运动粘度
    Ra = g * alpha * 70.0 * L_char**3 / (nu_ref * alpha_ref)
    print(f"瑞利数 Ra = {Ra:.2e}")
    
    # 特征速度（基于自然对流尺度）
    U_char = np.sqrt(g * alpha * 70.0 * L_char)
    
    # 特征时间（基于热扩散）
    t_char = L_char**2 / alpha_ref
    
    # 无量纲数
    Pr = nu_ref / alpha_ref  # 普朗特数
    Ste = cp_l * 70.0 / L    # 斯蒂芬数
    
    print(f"特征尺度: L={L_char:.4f}m, U={U_char:.4f}m/s, t={t_char:.1f}s")
    print(f"无量纲数: Pr={Pr:.3f}, Ste={Ste:.3f}")
    
    return {
        'L_char': L_char,
        'U_char': U_char,
        't_char': t_char,
        'T_char': 70.0,
        'p_char': rho_l * U_char**2,
        'Ra': Ra,
        'Pr': Pr,
        'Ste': Ste
    }

char_scales = compute_characteristic_scales()
L_char = char_scales['L_char']
U_char = char_scales['U_char']
t_char = char_scales['t_char']
T_char = char_scales['T_char']
p_char = char_scales['p_char']

# ==================== 拓扑优化参数 ====================
phi_total = 0.3  # 高导热材料体积比约束
case = 3         # 优化目标选择：1=平均温度，2=温度均方差，3=多目标
w1, w2, w3 = 1.0, 1.0, 1.0  # 多目标权重

# ==================== 采样参数 ====================
N_mass = 8000    # 质量守恒方程采样点数量
N_mom = 8000     # 动量方程采样点数量
N_energy = 8000  # 能量方程采样点数量
N_IC = 5000      # 初始条件采样点数量
N_BC1 = 2000     # 内管壁边界采样点数量
N_BC2 = 2000     # 外壳边界采样点数量
N_rho = 10000    # 拓扑设计变量采样点数量
N_obj = 10000    # 优化目标采样点数量

# ==================== 数值稳定性参数 ====================
eps = 1e-8       # 防止除零的小量
Am = 1e5         # 糊状区常数（达西阻尼系数）
xi = 1.0         # 粘度函数常数
mushy_width = DeltaT  # 糊状区宽度

# ==================== 残差块 ====================
class ResidualBlock(nn.Module):
    """残差块：缓解深层网络梯度消失，提升表达能力"""
    def __init__(self, dim, activation='tanh'):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        
        if activation == 'tanh':
            self.act = nn.Tanh()
        elif activation == 'gelu':
            self.act = nn.GELU()
        else:
            self.act = nn.SiLU()
        
        # 初始化权重
        nn.init.xavier_uniform_(self.fc1.weight, gain=0.5)
        nn.init.xavier_uniform_(self.fc2.weight, gain=0.5)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)
    
    def forward(self, x):
        residual = x
        out = self.act(self.fc1(x))
        out = self.fc2(out)
        return self.act(out + residual)

# ==================== 相变模型 ====================
class PhaseChangeModel:
    """数值稳定的相变模型"""
    
    def __init__(self, Tpc, DeltaT, L, hysteresis=1.0):
        self.Tpc = Tpc
        self.DeltaT = DeltaT
        self.L = L
        self.hysteresis = hysteresis  # 过冷/过热滞后
        
        # 相变温度区间
        self.T_melt = Tpc + hysteresis/2  # 熔化温度
        self.T_freeze = Tpc - hysteresis/2  # 凝固温度
        
        # 数值稳定性参数
        self.exp_clip = 50.0
        self.eps = 1e-10
        
    def compute_liquid_fraction(self, T, heating=True):
        """
        计算液相率φ(T)
        使用误差函数实现平滑过渡
        """
        if heating:
            # 加热过程
            T_center = self.T_melt
        else:
            # 冷却过程（考虑过冷）
            T_center = self.T_freeze
            
        # 归一化温度
        x = (T - T_center) / (self.DeltaT / 2)
        
        # 裁剪防止溢出
        x_clipped = torch.clamp(x, -self.exp_clip, self.exp_clip)
        
        # 使用sigmoid（误差函数的近似）
        phi = torch.sigmoid(2.5 * x_clipped)
        
        # 确保在[0,1]范围内
        return torch.clamp(phi, 0.0, 1.0)
    
    def compute_effective_cp(self, T, cp_s, cp_l, heating=True):
        """计算有效比热容（包含潜热）"""
        phi = self.compute_liquid_fraction(T, heating)
        
        # 计算温度的概率密度函数（高斯分布）
        sigma = self.DeltaT / 4.0
        exponent = -((T - self.Tpc) ** 2) / (2 * sigma ** 2 + self.eps)
        exponent_clipped = torch.clamp(exponent, -self.exp_clip, self.exp_clip)
        
        # 高斯核函数
        gaussian = torch.exp(exponent_clipped) / (sigma * np.sqrt(2 * np.pi) + self.eps)
        gaussian = torch.clamp(gaussian, 0.0, 1e3)
        
        # 有效比热容 = 显热 + 潜热
        cp_eff = cp_s + (cp_l - cp_s) * phi + self.L * gaussian
        
        return cp_eff, phi
    
    def compute_mushy_zone_damping(self, phi):
        """计算糊状区达西阻尼系数"""
        # 达西阻尼模型：S = Am * (1-φ)²/(φ³ + ε)
        phi_clamped = torch.clamp(phi, 0.01, 0.99)
        numerator = (1.0 - phi_clamped) ** 2
        denominator = phi_clamped ** 3 + self.eps
        
        S_t = Am * numerator / denominator
        return torch.clamp(S_t, 0.0, 1e10)

# ==================== 拓扑设计变量 ====================
class TopologyDesignVariable(nn.Module):
    """拓扑设计变量（SIMP方法实现）"""
    def __init__(self, penalty=3.0, filter_radius=0.02, volume_target=0.3):
        super().__init__()
        self.penalty = penalty  # SIMP惩罚因子
        self.filter_radius = filter_radius
        self.volume_target = volume_target
        
        # 设计变量网络
        self.design_net = nn.Sequential(
            nn.Linear(2, 128),
            nn.Tanh(),
            nn.Linear(128, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        
        # 初始化设计变量网络
        self._init_weights()
        
        # 拉格朗日乘子（用于体积约束）
        self.lagrange_multiplier = nn.Parameter(torch.tensor(0.0))
        self.penalty_param = nn.Parameter(torch.tensor(1.0))
        
    def _init_weights(self):
        """初始化权重，使初始设计接近目标体积比"""
        for layer in self.design_net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight, gain=0.1)
                nn.init.constant_(layer.bias, 0.0)
        
        # 调整最后一层偏置，使初始输出接近目标体积比
        with torch.no_grad():
            last_layer = self.design_net[-1]
            # 使sigmoid(0) ≈ volume_target
            desired_bias = np.log(self.volume_target / (1 - self.volume_target + eps))
            last_layer.bias.fill_(desired_bias)
    
    def forward(self, x_space):
        """
        前向传播
        输入: x_space [batch, 2] 无量纲空间坐标
        输出: rho_physical [batch, 1] 物理设计变量 ∈ [0,1]
        """
        # 1. 原始设计变量
        rho_raw = self.design_net(x_space)
        
        # 2. Sigmoid激活（确保在0-1之间）
        rho = torch.sigmoid(rho_raw)
        
        # 3. SIMP惩罚：rho^p
        rho_penalized = rho ** self.penalty
        
        return rho_penalized, rho
    
    def compute_volume_constraint(self, rho, weights):
        """计算体积约束违反量"""
        volume = torch.sum(rho * weights) / (torch.sum(weights) + eps)
        violation = volume - self.volume_target
        return violation, volume
    
    def compute_constraint_loss(self, rho, weights):
        """计算约束损失（增广拉格朗日法）"""
        violation, volume = self.compute_volume_constraint(rho, weights)
        
        # 拉格朗日项
        lagrangian_term = self.lagrange_multiplier * violation
        
        # 惩罚项
        penalty_term = 0.5 * self.penalty_param * violation ** 2
        
        # 边界惩罚（确保rho在[0,1]内）
        bound_penalty = torch.mean(torch.relu(rho - 1.0) ** 2 + torch.relu(-rho) ** 2)
        
        return lagrangian_term + penalty_term + 1e-3 * bound_penalty, volume, violation
    
    def update_multipliers(self, violation):
        """更新拉格朗日乘子和惩罚参数"""
        with torch.no_grad():
            # 更新拉格朗日乘子：λ_{k+1} = λ_k + ρ * g(x)
            self.lagrange_multiplier += self.penalty_param * violation
            
            # 如果约束违反较大，增加惩罚参数
            if abs(violation.item()) > 0.05:
                self.penalty_param *= 1.5
            elif abs(violation.item()) < 0.01:
                self.penalty_param *= 0.9
            
            # 限制惩罚参数范围
            self.penalty_param.data.clamp_(0.1, 1e6)
    
    def compute_regularization(self, x_space, rho):
        """计算拓扑正则化项"""
        # 空间梯度正则化（平滑性）
        grad_rho = torch.autograd.grad(
            rho, x_space,
            grad_outputs=torch.ones_like(rho),
            create_graph=True,
            retain_graph=True
        )[0]
        
        grad_norm = torch.sum(grad_rho**2, dim=1, keepdim=True)
        smooth_loss = torch.mean(grad_norm)
        
        # 二值性正则化（鼓励rho接近0或1）
        binary_loss = torch.mean(rho * (1 - rho))
        
        return 0.01 * smooth_loss + 0.1 * binary_loss

# ==================== 材料属性插值 ====================
class MaterialInterpolation:
    """混合材料属性插值（Voigt-Reuss-Hill平均）"""
    
    @staticmethod
    def interpolate_density(rho, phi):
        """插值密度"""
        # 转换为张量，确保设备一致性
        rho_s_tensor = torch.tensor(rho_s, dtype=torch.float32).to(rho.device)
        rho_l_tensor = torch.tensor(rho_l, dtype=torch.float32).to(rho.device)
        rho_Cu_tensor = torch.tensor(rho_Cu, dtype=torch.float32).to(rho.device)
        
        rho_PCM = rho_s_tensor + (rho_l_tensor - rho_s_tensor) * phi
        return rho_Cu_tensor * rho + (1.0 - rho) * rho_PCM
    
    @staticmethod
    def interpolate_conductivity(rho, phi):
        """插值导热系数（VRH平均）"""
        # 转换为张量
        lambda_s_tensor = torch.tensor(lambda_s, dtype=torch.float32).to(rho.device)
        lambda_l_tensor = torch.tensor(lambda_l, dtype=torch.float32).to(rho.device)
        lambda_Cu_tensor = torch.tensor(lambda_Cu, dtype=torch.float32).to(rho.device)
        
        lambda_PCM = lambda_s_tensor + (lambda_l_tensor - lambda_s_tensor) * phi
        
        # Voigt上界（并联）
        lambda_V = lambda_Cu_tensor * rho + lambda_PCM * (1 - rho)
        
        # Reuss下界（串联）
        lambda_R_inv = rho/(lambda_Cu_tensor + eps) + (1 - rho)/(lambda_PCM + eps)
        lambda_R = 1.0 / (lambda_R_inv + eps)
        
        # Hill平均
        return 0.5 * (lambda_V + lambda_R)
    
    @staticmethod
    def interpolate_viscosity(rho, phi, S_t):
        """插值粘度"""
        # 获取设备上的张量
        mu_Cu_tensor = torch.tensor(mu_Cu, dtype=torch.float32).to(rho.device)
        mu_l_tensor = torch.tensor(mu_l, dtype=torch.float32).to(rho.device)
        
        mu_PCM = mu_l_tensor + S_t  # 液相粘度 + 糊状区阻尼
        mu_PCM = torch.clamp(mu_PCM, 1e-6, 1e6)
        
        # 混合粘度（对数平均更合适）
        log_mu = rho * torch.log(mu_Cu_tensor + eps) + (1 - rho) * torch.log(mu_PCM + eps)
        return torch.exp(log_mu)
    
    @staticmethod
    def interpolate_specific_heat(rho, phi, cp_eff):
        """插值比热容（质量加权平均）"""
        # 转换为张量
        rho_Cu_tensor = torch.tensor(rho_Cu, dtype=torch.float32).to(rho.device)
        cp_Cu_tensor = torch.tensor(cp_Cu, dtype=torch.float32).to(rho.device)
        rho_s_tensor = torch.tensor(rho_s, dtype=torch.float32).to(rho.device)
        rho_l_tensor = torch.tensor(rho_l, dtype=torch.float32).to(rho.device)
        
        # 各相质量
        mass_Cu = rho * rho_Cu_tensor
        mass_PCM = (1 - rho) * (rho_s_tensor + (rho_l_tensor - rho_s_tensor) * phi)
        mass_total = mass_Cu + mass_PCM + eps
        
        # 质量加权平均
        return (mass_Cu * cp_Cu_tensor + mass_PCM * cp_eff) / mass_total
    
    @staticmethod
    def compute_all_properties(T, rho, heating=True):
        """计算所有材料属性"""
        # 相变模型
        phase_model = PhaseChangeModel(Tpc, DeltaT, L)
        cp_eff, phi = phase_model.compute_effective_cp(T, cp_s, cp_l, heating)
        S_t = phase_model.compute_mushy_zone_damping(phi)
        
        # 计算各属性
        rho_total = MaterialInterpolation.interpolate_density(rho, phi)
        lambda_total = MaterialInterpolation.interpolate_conductivity(rho, phi)
        mu_total = MaterialInterpolation.interpolate_viscosity(rho, phi, S_t)
        cp_total = MaterialInterpolation.interpolate_specific_heat(rho, phi, cp_eff)
        
        # 运动粘度和热扩散率
        nu = mu_total / (rho_total + eps)
        a = lambda_total / (rho_total * cp_total + eps)
        
        # 浮力项（Boussinesq近似）
        rho_ref = rho_l
        beta = alpha
        F_B = -rho_ref * beta * g * (T - Tpc)
        
        return {
            'rho': rho_total,
            'lambda': lambda_total,
            'mu': mu_total,
            'cp': cp_total,
            'nu': nu,
            'a': a,
            'phi': phi,
            'S_t': S_t,
            'F_B': F_B,
            'rho_ref': rho_ref
        }

# ==================== 拓扑优化PINN ====================
class TopoPINN(nn.Module):
    def __init__(self, hidden_layers=6, hidden_dim=256, activation='tanh'):
        super(TopoPINN, self).__init__()
        
        # 主网络：预测流场和温度场 (x*, y*, τ*) → (u*, v*, p*, T*)
        self.main_net = self._build_mlp(
            input_dim=3, 
            output_dim=4, 
            hidden_layers=hidden_layers,
            hidden_dim=hidden_dim,
            activation=activation
        )
        
        # 拓扑设计网络
        self.topology = TopologyDesignVariable(penalty=3.0, volume_target=phi_total)
        
        # 相变模型
        self.phase_model = PhaseChangeModel(Tpc, DeltaT, L)
        
        # 材料插值
        self.material = MaterialInterpolation()
        
        # 无量纲化参数
        self.register_buffer('L_char', torch.tensor(L_char))
        self.register_buffer('U_char', torch.tensor(U_char))
        self.register_buffer('t_char', torch.tensor(t_char))
        self.register_buffer('T_char', torch.tensor(T_char))
        self.register_buffer('p_char', torch.tensor(p_char))
        
        # 参考温度
        self.register_buffer('T0', torch.tensor(290.0))  # 初始温度
        self.register_buffer('Tw_heat', torch.tensor(360.0))  # 储热时内壁温度
        self.register_buffer('Tw_cool', torch.tensor(290.0))  # 释热时内壁温度
    
    def _build_mlp(self, input_dim, output_dim, hidden_layers, hidden_dim, activation):
        """构建多层感知机"""
        layers = []
        
        # 输入层
        layers.append(nn.Linear(input_dim, hidden_dim))
        if activation == 'tanh':
            layers.append(nn.Tanh())
        elif activation == 'gelu':
            layers.append(nn.GELU())
        else:
            layers.append(nn.SiLU())
        
        # 隐藏层（残差块）
        for _ in range(hidden_layers):
            layers.append(ResidualBlock(hidden_dim, activation))
        
        # 输出层
        layers.append(nn.Linear(hidden_dim, output_dim))
        
        return nn.Sequential(*layers)
    
    def forward(self, x):
        """
        前向传播
        输入: x [batch, 3] - 无量纲坐标 (x*, y*, τ*)
        输出: u, v, p, T, rho
        """
        # 提取空间坐标（用于拓扑网络）
        x_space = x[:, 0:2]
        
        # 主网络输出（无量纲量）
        out = self.main_net(x)
        
        # 转换为有量纲量
        u = out[:, 0:1] * self.U_char  # u = u* * U_char
        v = out[:, 1:2] * self.U_char  # v = v* * U_char
        p = out[:, 2:3] * self.p_char  # p = p* * p_char
        T = out[:, 3:4] * self.T_char + self.T0  # T = T* * T_char + T0
        
        # 拓扑设计变量
        rho_physical, rho_raw = self.topology(x_space)
        
        return u, v, p, T, rho_physical
    
    def compute_pde_residuals(self, x, heat_storage=True, require_grad=False):
        """
        计算PDE残差
        require_grad: 是否需要计算梯度（验证阶段可以设为False）
        """
        if require_grad:
            x.requires_grad_(True)
        
        # 前向传播
        u, v, p, T, rho = self(x)
        
        # 计算材料属性
        props = MaterialInterpolation.compute_all_properties(T, rho, heat_storage)
        
        # 计算梯度
        gradients = self._compute_gradients(u, v, p, T, x, require_grad)
        
        # 计算残差
        residuals = self._compute_residuals(u, v, p, T, rho, props, gradients, x, require_grad)
        
        return residuals
    
    def _compute_gradients(self, u, v, p, T, x, require_grad=True):
        """计算所有需要的梯度"""
        if not require_grad:
            # 如果不需梯度，返回空字典
            return {
                'u': (torch.zeros_like(u), torch.zeros_like(u), torch.zeros_like(u)),
                'v': (torch.zeros_like(v), torch.zeros_like(v), torch.zeros_like(v)),
                'p': (torch.zeros_like(p), torch.zeros_like(p)),
                'T': (torch.zeros_like(T), torch.zeros_like(T), torch.zeros_like(T)),
                'grad_T': torch.zeros_like(x)
            }
        
        # 一阶梯度
        grad_u = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u),
                                     create_graph=True, retain_graph=True)[0]
        grad_v = torch.autograd.grad(v, x, grad_outputs=torch.ones_like(v),
                                     create_graph=True, retain_graph=True)[0]
        grad_p = torch.autograd.grad(p, x, grad_outputs=torch.ones_like(p),
                                     create_graph=True, retain_graph=True)[0]
        grad_T = torch.autograd.grad(T, x, grad_outputs=torch.ones_like(T),
                                     create_graph=True, retain_graph=True)[0]
        
        # 转换为实际导数（无量纲→有量纲）
        dudx = grad_u[:, 0:1] / self.L_char
        dudy = grad_u[:, 1:2] / self.L_char
        dudt = grad_u[:, 2:3] / self.t_char
        
        dvdx = grad_v[:, 0:1] / self.L_char
        dvdy = grad_v[:, 1:2] / self.L_char
        dvdt = grad_v[:, 2:3] / self.t_char
        
        dpdx = grad_p[:, 0:1] / self.L_char
        dpdy = grad_p[:, 1:2] / self.L_char
        
        dTdx = grad_T[:, 0:1] / self.L_char
        dTdy = grad_T[:, 1:2] / self.L_char
        dTdt = grad_T[:, 2:3] / self.t_char
        
        # 二阶导数（需要时再计算）
        gradients = {
            'u': (dudx, dudy, dudt),
            'v': (dvdx, dvdy, dvdt),
            'p': (dpdx, dpdy),
            'T': (dTdx, dTdy, dTdt),
            'grad_T': grad_T
        }
        
        return gradients
    
    def _compute_residuals(self, u, v, p, T, rho, props, gradients, x, require_grad=True):
        """计算PDE残差"""
        if not require_grad:
            # 如果不需梯度，返回零残差
            return {
                'mass': torch.zeros_like(u),
                'mom_x': torch.zeros_like(u),
                'mom_y': torch.zeros_like(u),
                'energy': torch.zeros_like(u)
            }
        
        # 提取梯度
        dudx, dudy, dudt = gradients['u']
        dvdx, dvdy, dvdt = gradients['v']
        dpdx, dpdy = gradients['p']
        dTdx, dTdy, dTdt = gradients['T']
        
        # 材料属性
        rho_total = props['rho']
        nu = props['nu']
        a = props['a']
        S_t = props['S_t']
        F_B = props['F_B']
        
        # ========== 质量守恒 ==========
        # 瞬态连续性方程: ∂ρ/∂t + ∇·(ρu) = 0
        # 对于不可压缩流体: ∇·u = 0
        mass_residual = dudx + dvdy
        
        # ========== 动量守恒 ==========
        # x方向: ∂u/∂t + u·∇u = -1/ρ ∂p/∂x + ν∇²u - S_t·u
        # 对流项
        conv_u = u * dudx + v * dudy
        
        # 压力项
        pressure_u = dpdx / rho_total
        
        # 粘性项（需要二阶导数）
        d2udx2 = torch.autograd.grad(dudx, x, grad_outputs=torch.ones_like(dudx),
                                      create_graph=True, retain_graph=True)[0][:, 0:1] / self.L_char
        d2udy2 = torch.autograd.grad(dudy, x, grad_outputs=torch.ones_like(dudy),
                                      create_graph=True, retain_graph=True)[0][:, 1:2] / self.L_char
        viscous_u = nu * (d2udx2 + d2udy2)
        
        # 糊状区阻尼项
        damping_u = S_t * u
        
        # 时间导数
        transient_u = dudt
        
        # x方向动量残差
        mom_x_residual = transient_u + conv_u + pressure_u - viscous_u + damping_u
        
        # y方向: ∂v/∂t + u·∇v = -1/ρ ∂p/∂y + ν∇²v - S_t·v + F_B/ρ
        conv_v = u * dvdx + v * dvdy
        pressure_v = dpdy / rho_total
        
        d2vdx2 = torch.autograd.grad(dvdx, x, grad_outputs=torch.ones_like(dvdx),
                                      create_graph=True, retain_graph=True)[0][:, 0:1] / self.L_char
        d2vdy2 = torch.autograd.grad(dvdy, x, grad_outputs=torch.ones_like(dvdy),
                                      create_graph=True, retain_graph=True)[0][:, 1:2] / self.L_char
        viscous_v = nu * (d2vdx2 + d2vdy2)
        
        damping_v = S_t * v
        
        # 浮力项（Boussinesq近似）
        buoyancy = F_B / rho_total
        
        # 时间导数
        transient_v = dvdt
        
        # y方向动量残差
        mom_y_residual = transient_v + conv_v + pressure_v - viscous_v + damping_v - buoyancy
        
        # ========== 能量守恒 ==========
        # ∂T/∂t + u·∇T = a∇²T
        conv_T = u * dTdx + v * dTdy
        
        d2Tdx2 = torch.autograd.grad(dTdx, x, grad_outputs=torch.ones_like(dTdx),
                                      create_graph=True, retain_graph=True)[0][:, 0:1] / self.L_char
        d2Tdy2 = torch.autograd.grad(dTdy, x, grad_outputs=torch.ones_like(dTdy),
                                      create_graph=True, retain_graph=True)[0][:, 1:2] / self.L_char
        diffusion_T = a * (d2Tdx2 + d2Tdy2)
        
        # 时间导数
        transient_T = dTdt
        
        # 能量残差
        energy_residual = transient_T + conv_T - diffusion_T
        
        return {
            'mass': mass_residual,
            'mom_x': mom_x_residual,
            'mom_y': mom_y_residual,
            'energy': energy_residual
        }

# ==================== 采样策略 ====================
class SpaceTimeSampler:
    """智能时空采样器"""
    
    def __init__(self, r0, r1, t_max, L_char, t_char):
        self.r0 = r0
        self.r1 = r1
        self.t_max = t_max
        self.L_char = L_char
        self.t_char = t_char
        
        # 采样历史
        self.residual_history = []
        self.adaptivity_factor = 0.3  # 自适应采样比例
    
    def sample_uniform(self, N, time_stratified=True):
        """均匀采样"""
        # 空间采样（环形域面积均匀）
        N_total = N
        
        # 使用拉丁超立方采样确保均匀性
        sampler = qmc.LatinHypercube(d=2)
        sample = sampler.random(N_total)
        
        # 转换为极坐标
        r_sq = sample[:, 0] * (self.r1**2 - self.r0**2) + self.r0**2
        r = np.sqrt(r_sq)
        theta = sample[:, 1] * 2 * np.pi
        
        # 转换为直角坐标
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        
        # 时间采样
        if time_stratified:
            # 分层时间采样：相变期（中间时段）加密
            t_center = self.t_max / 2
            t_width = self.t_max / 4
            
            # 70%的点在相变期，30%在前后期
            N_center = int(N_total * 0.7)
            N_edge = N_total - N_center
            
            t_center_samples = np.random.normal(t_center, t_width/3, N_center)
            t_edge_samples = np.concatenate([
                np.random.uniform(0, t_center - t_width, N_edge//2),
                np.random.uniform(t_center + t_width, self.t_max, N_edge//2)
            ])
            
            t = np.concatenate([t_center_samples, t_edge_samples])
            np.random.shuffle(t)
        else:
            # 均匀时间采样
            t = np.random.uniform(0, self.t_max, N_total)
        
        # 无量纲化
        x_star = x / self.L_char
        y_star = y / self.L_char
        tau_star = t / self.t_char
        
        # 组合
        points = np.column_stack([x_star, y_star, tau_star])
        weights = np.ones((N_total, 1))  # 均匀权重
        
        return torch.tensor(points, dtype=torch.float32), torch.tensor(weights, dtype=torch.float32)
    
    def sample_boundary(self, N, boundary_type='inner', time_stratified=True):
        """边界采样"""
        if boundary_type == 'inner':
            r = self.r0
        else:  # 'outer'
            r = self.r1
            
        # 角度均匀采样
        theta = np.random.uniform(0, 2*np.pi, N)
        
        # 转换为直角坐标
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        
        # 时间采样
        if time_stratified:
            # 边界条件随时间变化，初期和末期采样更密集
            t = self._sample_boundary_time(N)
        else:
            t = np.random.uniform(0, self.t_max, N)
        
        # 无量纲化
        x_star = x / self.L_char
        y_star = y / self.L_char
        tau_star = t / self.t_char
        
        points = np.column_stack([x_star, y_star, tau_star])
        
        return torch.tensor(points, dtype=torch.float32)
    
    def _sample_boundary_time(self, N):
        """边界时间采样（考虑物理过程）"""
        # 边界条件变化主要集中在前期
        t_early = np.random.exponential(self.t_max/5, N//2)
        t_early = np.clip(t_early, 0, self.t_max)
        
        # 其余时间均匀采样
        t_late = np.random.uniform(0, self.t_max, N - N//2)
        
        t = np.concatenate([t_early, t_late])
        np.random.shuffle(t)
        
        return t
    
    def sample_initial(self, N):
        """初始条件采样"""
        # 空间均匀采样
        sampler = qmc.LatinHypercube(d=2)
        sample = sampler.random(N)
        
        r_sq = sample[:, 0] * (self.r1**2 - self.r0**2) + self.r0**2
        r = np.sqrt(r_sq)
        theta = sample[:, 1] * 2 * np.pi
        
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        
        # 时间固定为0
        t = np.zeros(N)
        
        # 无量纲化
        x_star = x / self.L_char
        y_star = y / self.L_char
        tau_star = t / self.t_char
        
        points = np.column_stack([x_star, y_star, tau_star])
        weights = np.ones((N, 1)) * r[:, np.newaxis]  # 径向权重
        
        return torch.tensor(points, dtype=torch.float32), torch.tensor(weights, dtype=torch.float32)

# ==================== 损失计算 ====================
class LossCalculator:
    """损失计算器"""
    
    def __init__(self, model, sampler):
        self.model = model
        self.sampler = sampler
        
        # 损失权重（自适应）
        self.weights = {
            'pde': 1.0,
            'bc': 10.0,
            'ic': 10.0,
            'topo': 100.0,
            'objective': 0.1
        }
        
        # 损失历史
        self.loss_history = {k: [] for k in self.weights.keys()}
        self.adaptive_update_freq = 100
        
        # 跟踪体积约束
        self.volume_violation = 0.0
        self.current_volume = 0.0
    
    def compute_total_loss(self, heat_storage=True, compute_objective=True, stage="joint"):
        """计算总损失"""
        # 采样点
        x_col, w_col = self.sampler.sample_uniform(N_mass)
        x_ic, w_ic = self.sampler.sample_initial(N_IC)
        x_bc1 = self.sampler.sample_boundary(N_BC1, 'inner')
        x_bc2 = self.sampler.sample_boundary(N_BC2, 'outer')
        
        # 移至设备
        x_col = x_col.to(device)
        w_col = w_col.to(device)
        x_ic = x_ic.to(device)
        w_ic = w_ic.to(device)
        x_bc1 = x_bc1.to(device)
        x_bc2 = x_bc2.to(device)
        
        # 计算各项损失
        losses = {}
        
        # PDE损失
        losses['pde'] = self.compute_pde_loss(x_col, w_col, heat_storage)
        
        # 初始条件损失
        losses['ic'] = self.compute_ic_loss(x_ic, w_ic, heat_storage)
        
        # 边界条件损失
        losses['bc'] = self.compute_bc_loss(x_bc1, x_bc2, heat_storage)
        
        # 拓扑约束损失
        losses['topo'] = self.compute_topo_loss(x_col, w_col)
        
        # 优化目标损失
        if compute_objective:
            losses['objective'] = self.compute_objective_loss()
        else:
            losses['objective'] = torch.tensor(0.0).to(device)
        
        # 自适应调整权重
        if stage != "pretrain":
            self.update_weights(losses)
        
        # 计算加权总损失
        total_loss = torch.tensor(0.0).to(device)
        for key, loss in losses.items():
            total_loss += self.weights[key] * loss
        
        # 记录损失历史
        for key in losses:
            self.loss_history[key].append(losses[key].item())
        
        return total_loss, losses
    
    def compute_pde_loss(self, x, w, heat_storage):
        """计算PDE残差损失"""
        residuals = self.model.compute_pde_residuals(x, heat_storage, require_grad=True)
        
        # 加权残差平方和
        loss = torch.mean(residuals['mass'] ** 2 * w) + \
               torch.mean(residuals['mom_x'] ** 2 * w) + \
               torch.mean(residuals['mom_y'] ** 2 * w) + \
               torch.mean(residuals['energy'] ** 2 * w)
        
        return loss
    
    def compute_ic_loss(self, x, w, heat_storage):
        """计算初始条件损失"""
        T0 = self.model.T0 if heat_storage else self.model.Tw_heat
        
        u, v, p, T, rho = self.model(x)
        
        # 初始温度应为T0，速度应为0
        T_loss = torch.mean((T - T0) ** 2 * w)
        u_loss = torch.mean(u ** 2 * w)
        v_loss = torch.mean(v ** 2 * w)
        
        return T_loss + u_loss + v_loss
    
    def compute_bc_loss(self, x_inner, x_outer, heat_storage):
        """计算边界条件损失"""
        # 内壁边界（恒温）
        Tw = self.model.Tw_heat if heat_storage else self.model.Tw_cool
        
        u_inner, v_inner, p_inner, T_inner, rho_inner = self.model(x_inner)
        T_loss_inner = torch.mean((T_inner - Tw) ** 2)
        
        # 外壁边界（绝热近似）
        u_outer, v_outer, p_outer, T_outer, rho_outer = self.model(x_outer)
        
        # 计算法向温度梯度
        x_outer.requires_grad_(True)
        T_outer = self.model(x_outer)[3]
        
        grad_T = torch.autograd.grad(T_outer, x_outer, 
                                     grad_outputs=torch.ones_like(T_outer),
                                     create_graph=True)[0]
        
        dTdx = grad_T[:, 0:1] / self.model.L_char
        dTdy = grad_T[:, 1:2] / self.model.L_char
        
        # 计算径向单位向量
        x_real = x_outer[:, 0:1] * self.model.L_char
        y_real = x_outer[:, 1:2] * self.model.L_char
        r = torch.sqrt(x_real**2 + y_real**2 + eps)
        
        # 法向梯度 ∂T/∂n = ∇T·n
        n_x = x_real / r
        n_y = y_real / r
        dTdn = n_x * dTdx + n_y * dTdy
        
        T_loss_outer = torch.mean(dTdn ** 2)
        
        return T_loss_inner + T_loss_outer
    
    def compute_topo_loss(self, x, w):
        """计算拓扑约束损失"""
        x_space = x[:, 0:2]
        rho_physical, rho_raw = self.model.topology(x_space)
        
        # 体积约束损失
        constraint_loss, volume, violation = self.model.topology.compute_constraint_loss(rho_physical, w)
        
        # 正则化损失
        reg_loss = self.model.topology.compute_regularization(x_space, rho_physical)
        
        # 更新乘子（在训练循环中）
        self.volume_violation = violation.item()
        self.current_volume = volume.item()
        
        return constraint_loss + reg_loss
    
    def compute_objective_loss(self):
        """计算优化目标损失"""
        # 采样点用于计算目标函数
        x_obj, w_obj = self.sampler.sample_uniform(N_obj)
        x_obj = x_obj.to(device)
        w_obj = w_obj.to(device)
        
        # 前向传播
        u, v, p, T, rho = self.model(x_obj)
        
        if case == 1:
            # Case 1: 最小化平均温度
            T_avg = torch.sum(T * w_obj) / torch.sum(w_obj)
            return T_avg ** 2
            
        elif case == 2:
            # Case 2: 最小化温度均方差
            T_avg = torch.sum(T * w_obj) / torch.sum(w_obj)
            T_var = torch.sum((T - T_avg) ** 2 * w_obj) / torch.sum(w_obj)
            return T_var
            
        else:
            # Case 3: 多目标优化
            # 平均温度
            T_avg = torch.sum(T * w_obj) / torch.sum(w_obj)
            L_avg = T_avg ** 2
            
            # 温度均方差
            T_var = torch.sum((T - T_avg) ** 2 * w_obj) / torch.sum(w_obj)
            L_var = T_var
            
            # 火积耗散（修正版本）
            x_obj.requires_grad_(True)
            T = self.model(x_obj)[3]
            
            # 计算温度梯度
            grad_T = torch.autograd.grad(T, x_obj, 
                                         grad_outputs=torch.ones_like(T),
                                         create_graph=True)[0]
            dTdx = grad_T[:, 0:1] / self.model.L_char
            dTdy = grad_T[:, 1:2] / self.model.L_char
            
            # 温度梯度模长平方
            grad_T_sq = dTdx**2 + dTdy**2
            
            # 计算导热系数
            rho_design = self.model.topology(x_obj[:, 0:2])[0]
            props = MaterialInterpolation.compute_all_properties(T, rho_design, True)
            lambda_total = props['lambda']
            
            # 火积耗散率：λ|∇T|²
            entransy_diss = torch.sum(lambda_total * grad_T_sq * w_obj) / torch.sum(w_obj)
            L_entransy = entransy_diss ** 2
            
            # 加权和
            return w1 * L_avg + w2 * L_var + w3 * L_entransy
    
    def update_weights(self, losses):
        """自适应调整损失权重"""
        # 每100步更新一次
        if len(self.loss_history['pde']) % self.adaptive_update_freq != 0:
            return
        
        # 计算最近损失的统计量
        recent_losses = {}
        for key in losses.keys():
            if len(self.loss_history[key]) >= self.adaptive_update_freq:
                recent = self.loss_history[key][-self.adaptive_update_freq:]
                recent_losses[key] = np.mean(recent)
            else:
                recent_losses[key] = losses[key].item()
        
        # 基于相对量级调整权重
        total = sum(recent_losses.values())
        if total > 0:
            for key in recent_losses:
                target_ratio = 1.0 / len(recent_losses)  # 目标：各损失贡献相等
                current_ratio = recent_losses[key] / total
                
                if current_ratio > 0:
                    # 调整权重：权重 ∝ 目标比例 / 当前比例
                    self.weights[key] *= target_ratio / (current_ratio + eps)
        
        # 权重裁剪
        for key in self.weights:
            self.weights[key] = np.clip(self.weights[key], 1e-3, 1e3)

# ==================== 训练器 ====================
class TopoPINNTrainer:
    """拓扑优化PINN训练器"""
    
    def __init__(self, model, sampler, loss_calculator):
        self.model = model
        self.sampler = sampler
        self.loss_calculator = loss_calculator
        
        # 训练阶段配置
        self.stages = [
            {'name': 'pretrain', 'epochs': 2000, 'lr': 1e-3, 'freeze_topo': True, 'compute_obj': False},
            {'name': 'topo', 'epochs': 1500, 'lr': 1e-3, 'freeze_main': True, 'compute_obj': False},
            {'name': 'joint', 'epochs': 2000, 'lr': 5e-4, 'freeze_none': True, 'compute_obj': False},
            {'name': 'fine', 'epochs': 1000, 'lr': 1e-4, 'freeze_none': True, 'compute_obj': True}
        ]
        
        # 训练历史
        self.history = {
            'loss': [],
            'volume': [],
            'violation': [],
            'stage': []
        }
        
        # 创建保存目录
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.save_dir = f"./results/case{case}_{timestamp}"
        os.makedirs(self.save_dir, exist_ok=True)
        
    def train(self):
        """执行训练"""
        print("=" * 60)
        print(f"开始训练拓扑优化PINN (Case {case})")
        print("=" * 60)
        
        current_epoch = 0
        
        for stage_idx, stage_config in enumerate(self.stages):
            stage_name = stage_config['name']
            epochs = stage_config['epochs']
            
            print(f"\n{'='*60}")
            print(f"阶段 {stage_idx+1}: {stage_name.upper()}")
            print(f"轮次: {epochs}, 学习率: {stage_config['lr']}")
            print(f"{'='*60}")
            
            # 设置参数冻结
            self._set_parameter_freezing(stage_config)
            
            # 创建优化器
            trainable_params = filter(lambda p: p.requires_grad, self.model.parameters())
            optimizer = optim.Adam(trainable_params, lr=stage_config['lr'])
            
            # 学习率调度器
            scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=stage_config['lr']/100)
            
            # 阶段训练
            for epoch in range(epochs):
                current_epoch += 1
                
                # 计算损失
                total_loss, losses = self.loss_calculator.compute_total_loss(
                    heat_storage=True,
                    compute_objective=stage_config['compute_obj'],
                    stage=stage_name
                )
                
                # 反向传播
                optimizer.zero_grad()
                total_loss.backward()
                
                # 梯度裁剪
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                
                # 优化步骤
                optimizer.step()
                scheduler.step()
                
                # 更新拓扑约束乘子
                if stage_name in ['topo', 'joint', 'fine']:
                    violation = self.loss_calculator.volume_violation
                    self.model.topology.update_multipliers(torch.tensor(violation))
                
                # 记录历史
                self.history['loss'].append(total_loss.item())
                self.history['volume'].append(self.loss_calculator.current_volume)
                self.history['violation'].append(self.loss_calculator.volume_violation)
                self.history['stage'].append(stage_idx)
                
                # 输出进度
                if (epoch + 1) % 100 == 0:
                    print(f"Epoch {current_epoch:4d} | Loss: {total_loss.item():.4e} | "
                          f"Volume: {self.loss_calculator.current_volume:.4f} | "
                          f"Violation: {self.loss_calculator.volume_violation:.4e}")
                
                # 保存检查点
                if (epoch + 1) % 500 == 0:
                    self.save_checkpoint(current_epoch, stage_name)
            
            # 阶段结束，保存模型
            self.save_checkpoint(current_epoch, f"stage_{stage_name}_final")
        
        print("\n训练完成!")
        self.save_final_results()
    
    def _set_parameter_freezing(self, stage_config):
        """设置参数冻结"""
        for name, param in self.model.named_parameters():
            if stage_config.get('freeze_topo', False) and 'topology' in name:
                param.requires_grad = False
            elif stage_config.get('freeze_main', False) and 'topology' not in name:
                param.requires_grad = False
            else:
                param.requires_grad = True
    
    def save_checkpoint(self, epoch, tag):
        """保存检查点"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'history': self.history,
            'char_scales': char_scales
        }
        
        filename = f"{self.save_dir}/checkpoint_{tag}_epoch{epoch}.pth"
        torch.save(checkpoint, filename)
        
        # 保存配置
        config = {
            'case': case,
            'phi_total': phi_total,
            'weights': [w1, w2, w3],
            'model_config': {
                'hidden_layers': 6,
                'hidden_dim': 256
            }
        }
        
        with open(f"{self.save_dir}/config.json", 'w') as f:
            json.dump(config, f, indent=2)
    
    def save_final_results(self):
        """保存最终结果"""
        # 保存模型
        torch.save(self.model.state_dict(), f"{self.save_dir}/model_final.pth")
        
        # 保存训练历史
        np.savez(f"{self.save_dir}/training_history.npz",
                 loss=self.history['loss'],
                 volume=self.history['volume'],
                 violation=self.history['violation'],
                 stage=self.history['stage'])
        
        # 绘制训练曲线
        self.plot_training_history()
        
        print(f"\n所有结果已保存到: {self.save_dir}")
    
    def plot_training_history(self):
        """绘制训练历史曲线"""
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        
        # 总损失
        axes[0, 0].semilogy(self.history['loss'])
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Total Loss')
        axes[0, 0].set_title('Total Loss History')
        axes[0, 0].grid(True, alpha=0.3)
        
        # 体积分数
        axes[0, 1].plot(self.history['volume'])
        axes[0, 1].axhline(y=phi_total, color='r', linestyle='--', label='Target')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Volume Fraction')
        axes[0, 1].set_title('Volume Constraint')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # 约束违反
        axes[1, 0].semilogy(np.abs(self.history['violation']))
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Constraint Violation')
        axes[1, 0].set_title('Constraint Violation History')
        axes[1, 0].grid(True, alpha=0.3)
        
        # 训练阶段
        axes[1, 1].plot(self.history['stage'])
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Stage')
        axes[1, 1].set_title('Training Stage Progression')
        axes[1, 1].set_yticks([0, 1, 2, 3])
        axes[1, 1].set_yticklabels(['Pretrain', 'Topo', 'Joint', 'Fine'])
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{self.save_dir}/training_history.png", dpi=300)
        plt.close()

# ==================== 可视化 ====================
def visualize_results(model, save_dir="./results/visualization"):
    """可视化结果"""
    os.makedirs(save_dir, exist_ok=True)
    
    # 生成网格
    r = np.linspace(r0, r1, 100)
    theta = np.linspace(0, 2*np.pi, 100)
    R, Theta = np.meshgrid(r, theta)
    X = R * np.cos(Theta)
    Y = R * np.sin(Theta)
    
    # 关键时间点
    tau_list = [0.1, 0.3, 0.5, 0.7, 1.0]  # 无量纲时间
    
    for tau_star in tau_list:
        tau = tau_star * t_char  # 转换为实际时间
        
        # 准备输入数据
        x_flat = X.flatten()[:, np.newaxis] / L_char
        y_flat = Y.flatten()[:, np.newaxis] / L_char
        t_flat = np.full_like(x_flat, tau_star)
        
        x_input = np.hstack([x_flat, y_flat, t_flat])
        x_tensor = torch.tensor(x_input, dtype=torch.float32).to(device)
        
        # 前向传播
        with torch.no_grad():
            u, v, p, T, rho = model(x_tensor)
            
            # 计算液相率
            phase_model = PhaseChangeModel(Tpc, DeltaT, L)
            phi = phase_model.compute_liquid_fraction(T.cpu(), heating=True)
        
        # 重塑为网格
        T_grid = T.cpu().numpy().reshape(100, 100)
        phi_grid = phi.numpy().reshape(100, 100)
        rho_grid = rho.cpu().numpy().reshape(100, 100)
        
        # 计算速度幅值
        u_mag = torch.sqrt(u**2 + v**2).cpu().numpy().reshape(100, 100)
        
        # 绘图
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        
        # 温度场
        im1 = axes[0, 0].contourf(X, Y, T_grid, levels=50, cmap='jet', vmin=290, vmax=360)
        axes[0, 0].set_title(f'Temperature (τ={tau:.0f}s)')
        axes[0, 0].set_xlabel('x (m)')
        axes[0, 0].set_ylabel('y (m)')
        axes[0, 0].axis('equal')
        plt.colorbar(im1, ax=axes[0, 0])
        
        # 液相率
        im2 = axes[0, 1].contourf(X, Y, phi_grid, levels=50, cmap='viridis', vmin=0, vmax=1)
        axes[0, 1].set_title(f'Liquid Fraction (τ={tau:.0f}s)')
        axes[0, 1].set_xlabel('x (m)')
        axes[0, 1].set_ylabel('y (m)')
        axes[0, 1].axis('equal')
        plt.colorbar(im2, ax=axes[0, 1])
        
        # 拓扑结构
        im3 = axes[0, 2].contourf(X, Y, rho_grid, levels=50, cmap='binary', vmin=0, vmax=1)
        axes[0, 2].set_title(f'Topology (τ={tau:.0f}s)')
        axes[0, 2].set_xlabel('x (m)')
        axes[0, 2].set_ylabel('y (m)')
        axes[0, 2].axis('equal')
        plt.colorbar(im3, ax=axes[0, 2])
        
        # 速度场
        im4 = axes[1, 0].contourf(X, Y, u_mag, levels=50, cmap='plasma')
        axes[1, 0].set_title(f'Velocity Magnitude (τ={tau:.0f}s)')
        axes[1, 0].set_xlabel('x (m)')
        axes[1, 0].set_ylabel('y (m)')
        axes[1, 0].axis('equal')
        plt.colorbar(im4, ax=axes[1, 0])
        
        # 速度矢量（采样显示）
        skip = 10
        u_vec = u.cpu().numpy().reshape(100, 100)
        v_vec = v.cpu().numpy().reshape(100, 100)
        axes[1, 1].quiver(X[::skip, ::skip], Y[::skip, ::skip], 
                          u_vec[::skip, ::skip], v_vec[::skip, ::skip])
        axes[1, 1].set_title(f'Velocity Vectors (τ={tau:.0f}s)')
        axes[1, 1].set_xlabel('x (m)')
        axes[1, 1].set_ylabel('y (m)')
        axes[1, 1].axis('equal')
        
        # 温度剖面
        theta_idx = 0  # 角度索引
        axes[1, 2].plot(r, T_grid[theta_idx, :], 'b-', label=f'θ={theta_idx*3.6}°')
        axes[1, 2].plot(r, T_grid[25, :], 'g-', label='θ=90°')
        axes[1, 2].plot(r, T_grid[50, :], 'r-', label='θ=180°')
        axes[1, 2].set_xlabel('Radius (m)')
        axes[1, 2].set_ylabel('Temperature (K)')
        axes[1, 2].set_title('Radial Temperature Profiles')
        axes[1, 2].legend()
        axes[1, 2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{save_dir}/results_tau_{tau:.0f}s.png", dpi=300)
        plt.close()
    
    print(f"可视化结果已保存到 {save_dir}")

# ==================== 验证函数 ====================
def validate_model(model):
    """验证模型物理正确性"""
    print("\n" + "="*60)
    print("模型验证")
    print("="*60)
    
    # 创建采样器
    sampler = SpaceTimeSampler(r0, r1, t_char, L_char, t_char)
    
    # 采样验证点
    x_val, w_val = sampler.sample_uniform(1000)
    x_val = x_val.to(device)
    w_val = w_val.to(device)
    
    # 验证阶段临时启用梯度计算
    with torch.enable_grad():
        # 计算残差
        residuals = model.compute_pde_residuals(x_val, heat_storage=True, require_grad=True)
        
        # 计算平均残差
        mass_res = torch.mean(residuals['mass'] ** 2).item()
        mom_x_res = torch.mean(residuals['mom_x'] ** 2).item()
        mom_y_res = torch.mean(residuals['mom_y'] ** 2).item()
        energy_res = torch.mean(residuals['energy'] ** 2).item()
        
        print(f"质量守恒残差: {mass_res:.2e}")
        print(f"x动量守恒残差: {mom_x_res:.2e}")
        print(f"y动量守恒残差: {mom_y_res:.2e}")
        print(f"能量守恒残差: {energy_res:.2e}")
        
        # 验证体积约束
        x_space = x_val[:, 0:2]
        rho_physical, _ = model.topology(x_space)
        volume = torch.sum(rho_physical * w_val) / torch.sum(w_val)
        print(f"体积分数: {volume.item():.4f} (目标: {phi_total})")
        
        # 验证边界条件
        x_bc = sampler.sample_boundary(500, 'inner')
        x_bc = x_bc.to(device)
        u, v, p, T, rho = model(x_bc)
        T_error = torch.mean((T - 360.0) ** 2).item()
        print(f"内壁温度误差: {T_error:.2e}")
        
        # 验证初始条件
        x_ic, w_ic = sampler.sample_initial(500)
        x_ic = x_ic.to(device)
        u, v, p, T, rho = model(x_ic)
        T_error = torch.mean((T - 290.0) ** 2).item()
        u_error = torch.mean(u ** 2).item()
        v_error = torch.mean(v ** 2).item()
        print(f"初始温度误差: {T_error:.2e}")
        print(f"初始速度误差: u={u_error:.2e}, v={v_error:.2e}")
    
    print("="*60)

# ==================== 主程序 ====================
if __name__ == "__main__":
    print("拓扑优化PINN - 相变储热系统")
    print("="*60)
    
    # 1. 初始化模型
    print("初始化模型...")
    model = TopoPINN(hidden_layers=6, hidden_dim=256, activation='tanh').to(device)
    
    # 2. 创建采样器
    sampler = SpaceTimeSampler(r0, r1, t_char, L_char, t_char)
    
    # 3. 创建损失计算器
    loss_calculator = LossCalculator(model, sampler)
    
    # 4. 创建训练器
    trainer = TopoPINNTrainer(model, sampler, loss_calculator)
    
    # 5. 验证初始模型
    validate_model(model)
    
    # 6. 训练模型
    trainer.train()
    
    # 7. 验证训练后模型
    validate_model(model)
    
    # 8. 可视化结果
    print("\n生成可视化结果...")
    visualize_results(model, save_dir=f"{trainer.save_dir}/visualization")
    
    print("\n" + "="*60)
    print("程序执行完成!")
    print(f"结果保存目录: {trainer.save_dir}")
    print("="*60)

使用设备: cuda
瑞利数 Ra = 3.24e+08
特征尺度: L=0.0800m, U=0.0741m/s, t=77499.7s
无量纲数: Pr=15.933, Ste=0.656
拓扑优化PINN - 相变储热系统
初始化模型...

模型验证
质量守恒残差: 1.92e-02
x动量守恒残差: 1.05e+16
y动量守恒残差: 1.22e+16
能量守恒残差: 5.30e+00
体积分数: 0.0270 (目标: 0.3)
内壁温度误差: 5.97e+03
初始温度误差: 5.75e+01
初始速度误差: u=5.23e-05, v=1.45e-05
开始训练拓扑优化PINN (Case 3)

阶段 1: PRETRAIN
轮次: 2000, 学习率: 0.001
Epoch  100 | Loss: 2.2768e+03 | Volume: 0.0270 | Violation: -2.7300e-01
Epoch  200 | Loss: 2.2809e+03 | Volume: 0.0270 | Violation: -2.7300e-01
Epoch  300 | Loss: 2.2571e+03 | Volume: 0.0270 | Violation: -2.7300e-01
Epoch  400 | Loss: 2.2168e+03 | Volume: 0.0270 | Violation: -2.7300e-01
Epoch  500 | Loss: 2.4622e+03 | Volume: 0.0270 | Violation: -2.7300e-01
Epoch  600 | Loss: 2.2997e+03 | Volume: 0.0270 | Violation: -2.7300e-01
Epoch  700 | Loss: 2.1878e+03 | Volume: 0.0270 | Violation: -2.7300e-01
Epoch  800 | Loss: 2.2229e+03 | Volume: 0.0270 | Violation: -2.7300e-01
Epoch  900 | Loss: 2.3458e+03 | Volume: 0.0270 | Violation: -2.7300e-01
Epoch